# Full MNIST Synthesis Benchmark

## Overview

Previous notebooks used a **10,000-sample subset** to keep experiments fast. This notebook is the final synthesis step: we retrain the main models on the **full MNIST dataset** and compare them under one shared protocol.

### Models Compared

| Model | Origin notebook | What changes |
|-------|-----------------|--------------|
| MLP | [deep-neural_network_pytorch](deep-neural_network_pytorch.ipynb) | Fully connected 784 → 128 → 64 → 10 |
| CNN | [deep-neural_network_cnn](deep-neural_network_cnn.ipynb) | Conv16 → Conv32 → FC128 → 10 |
| CNN + L2 | [deep-neural_network_regularization](deep-neural_network_regularization.ipynb) | Same CNN + weight decay |
| CNN + Dropout | [deep-neural_network_regularization](deep-neural_network_regularization.ipynb) | Same CNN + Dropout on dense head |

### Protocol

- Full MNIST: **60,000 train / 10,000 test** (canonical split)
- Optimizer: SGD, `lr=0.1`, batch size 128
- Epochs: 15 (enough to rank models on full data without a multi-hour run)
- Same seed before each training run

### Goal

Answer one question cleanly: **when more data is available, which modeling choice actually wins?**


## 1. Setup and Imports


In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.datasets import fetch_openml

print(f"PyTorch version: {torch.__version__}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

np.random.seed(42)
torch.manual_seed(42)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = [10, 6]
plt.rcParams['font.size'] = 12


## 2. Full MNIST Loading

We use the canonical OpenML split: the first 60,000 images for training and the last 10,000 for testing. This matches the official MNIST protocol and makes the synthesis comparable to common benchmarks.


In [ ]:
print("Loading full MNIST dataset...")
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')

X_full = mnist.data.astype(np.float32) / 255.0
y_full = mnist.target.astype(int)

# Canonical split
X_train, y_train = X_full[:60000], y_full[:60000]
X_test, y_test = X_full[60000:], y_full[60000:]

print(f"Train: {X_train.shape[0]:,} samples")
print(f"Test:  {X_test.shape[0]:,} samples")
print(f"Classes: {np.unique(y_full)}")


In [ ]:
BATCH_SIZE = 128

X_train_flat = torch.FloatTensor(X_train)
X_test_flat = torch.FloatTensor(X_test)
X_train_img = torch.FloatTensor(X_train.reshape(-1, 1, 28, 28))
X_test_img = torch.FloatTensor(X_test.reshape(-1, 1, 28, 28))
y_train_t = torch.LongTensor(y_train)
y_test_t = torch.LongTensor(y_test)

train_loader_mlp = DataLoader(TensorDataset(X_train_flat, y_train_t), batch_size=BATCH_SIZE, shuffle=True)
test_loader_mlp = DataLoader(TensorDataset(X_test_flat, y_test_t), batch_size=BATCH_SIZE, shuffle=False)
train_loader_cnn = DataLoader(TensorDataset(X_train_img, y_train_t), batch_size=BATCH_SIZE, shuffle=True)
test_loader_cnn = DataLoader(TensorDataset(X_test_img, y_test_t), batch_size=BATCH_SIZE, shuffle=False)

print(f"MLP batch: {next(iter(train_loader_mlp))[0].shape}")
print(f"CNN batch: {next(iter(train_loader_cnn))[0].shape}")
print(f"Train batches per epoch: {len(train_loader_mlp)}")


## 3. Model Definitions

We reuse the same architectures as in the previous notebooks. Nothing new is introduced here except the data scale.


In [ ]:
class MLPClassifier(nn.Module):
    """Fully connected baseline: 784 → 128 → 64 → 10"""

    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)


class CNNClassifier(nn.Module):
    """Conv16 → Pool → Conv32 → Pool → FC128 → 10"""

    def __init__(self, dropout_p=0.0):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(p=dropout_p),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        if x.ndim == 2:
            x = x.view(-1, 1, 28, 28)
        return self.classifier(self.features(x))


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


print(f"MLP params: {count_params(MLPClassifier()):,}")
print(f"CNN params: {count_params(CNNClassifier()):,}")


## 4. Training Utilities

Shared loop for every model. Train accuracy during epochs uses a light probe for speed. Final scoreboard metrics are always computed on the full train and test sets.


In [ ]:
def evaluate(model, data_loader, max_batches=None):
    model.eval()
    correct, total, loss_sum = 0, 0, 0.0
    criterion = nn.CrossEntropyLoss()

    with torch.no_grad():
        for i, (X_batch, y_batch) in enumerate(data_loader):
            if max_batches is not None and i >= max_batches:
                break
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            logits = model(X_batch)
            loss_sum += criterion(logits, y_batch).item() * y_batch.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)

    return loss_sum / total, correct / total


def train_model(
    model,
    train_loader,
    test_loader,
    name,
    lr=0.1,
    weight_decay=0.0,
    epochs=15,
    print_every=5,
):
    torch.manual_seed(42)
    model = model.to(device)
    # Re-init after moving and reseeding for fair comparison
    model.apply(lambda m: m.reset_parameters() if hasattr(m, 'reset_parameters') else None)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr, weight_decay=weight_decay)

    history = {
        'train_acc': [], 'test_acc': [], 'gap': [], 'train_loss': [], 'test_loss': []
    }

    print(f"\nTraining {name}...")
    print(f"  lr={lr}, weight_decay={weight_decay}, epochs={epochs}, batch_size={BATCH_SIZE}")
    print("=" * 70)
    t0 = time.time()

    for epoch in range(epochs):
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()

        train_loss, train_acc = evaluate(model, train_loader, max_batches=30)
        test_loss, test_acc = evaluate(model, test_loader)
        gap = train_acc - test_acc

        history['train_loss'].append(train_loss)
        history['test_loss'].append(test_loss)
        history['train_acc'].append(train_acc)
        history['test_acc'].append(test_acc)
        history['gap'].append(gap)

        if epoch % print_every == 0 or epoch == epochs - 1:
            print(
                f"Epoch {epoch + 1:2d}/{epochs}: "
                f"Train {train_acc:.2%} | Test {test_acc:.2%} | Gap {gap:+.2%}"
            )

    # Full final metrics
    final_train_loss, final_train_acc = evaluate(model, train_loader)
    final_test_loss, final_test_acc = evaluate(model, test_loader)
    history['train_loss'][-1] = final_train_loss
    history['train_acc'][-1] = final_train_acc
    history['test_loss'][-1] = final_test_loss
    history['test_acc'][-1] = final_test_acc
    history['gap'][-1] = final_train_acc - final_test_acc
    history['seconds'] = time.time() - t0
    history['params'] = count_params(model)

    print("=" * 70)
    print(
        f"Final ({name}): Train {final_train_acc:.2%} | "
        f"Test {final_test_acc:.2%} | Gap {history['gap'][-1]:+.2%} | "
        f"Time {history['seconds']:.1f}s"
    )
    return model, history


## 5. Run the Full-Data Benchmark

Four models, one shared recipe. Only architecture and regularization change.


In [ ]:
EPOCHS = 15
LR = 0.1
WEIGHT_DECAY = 1e-4
DROPOUT_P = 0.5

configs = [
    {
        'name': 'MLP',
        'builder': lambda: MLPClassifier(),
        'train_loader': train_loader_mlp,
        'test_loader': test_loader_mlp,
        'weight_decay': 0.0,
        'color': '#9b59b6',
    },
    {
        'name': 'CNN',
        'builder': lambda: CNNClassifier(dropout_p=0.0),
        'train_loader': train_loader_cnn,
        'test_loader': test_loader_cnn,
        'weight_decay': 0.0,
        'color': '#3498db',
    },
    {
        'name': 'CNN + L2',
        'builder': lambda: CNNClassifier(dropout_p=0.0),
        'train_loader': train_loader_cnn,
        'test_loader': test_loader_cnn,
        'weight_decay': WEIGHT_DECAY,
        'color': '#2ecc71',
    },
    {
        'name': 'CNN + Dropout',
        'builder': lambda: CNNClassifier(dropout_p=DROPOUT_P),
        'train_loader': train_loader_cnn,
        'test_loader': test_loader_cnn,
        'weight_decay': 0.0,
        'color': '#e67e22',
    },
]

results = {}
models = {}

for cfg in configs:
    torch.manual_seed(42)
    model, history = train_model(
        model=cfg['builder'](),
        train_loader=cfg['train_loader'],
        test_loader=cfg['test_loader'],
        name=cfg['name'],
        lr=LR,
        weight_decay=cfg['weight_decay'],
        epochs=EPOCHS,
        print_every=5,
    )
    results[cfg['name']] = history
    models[cfg['name']] = model


## 6. Learning Curves


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for cfg in configs:
    name = cfg['name']
    color = cfg['color']
    hist = results[name]
    axes[0].plot(hist['test_acc'], color=color, linewidth=2, label=name)
    axes[1].plot(hist['gap'], color=color, linewidth=2, label=name)

axes[0].set_title('Test Accuracy on Full MNIST')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Test accuracy')
axes[0].legend()

axes[1].set_title('Generalization Gap (Train - Test)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Gap')
axes[1].axhline(0, color='gray', linestyle=':')
axes[1].legend()

plt.tight_layout()
plt.show()


## 7. Final Scoreboard


In [ ]:
print("=" * 92)
print("FULL MNIST SYNTHESIS — FINAL RESULTS")
print("=" * 92)
print(
    f"{'Model':<16} {'Params':<10} {'Train':<10} {'Test':<10} "
    f"{'Gap':<10} {'Best Test':<12} {'Time (s)':<10}"
)
print("-" * 92)

scoreboard = []
for cfg in configs:
    name = cfg['name']
    hist = results[name]
    row = {
        'name': name,
        'params': hist['params'],
        'train': hist['train_acc'][-1],
        'test': hist['test_acc'][-1],
        'gap': hist['gap'][-1],
        'best_test': max(hist['test_acc']),
        'seconds': hist['seconds'],
        'color': cfg['color'],
    }
    scoreboard.append(row)
    print(
        f"{row['name']:<16} {row['params']:<10,} {row['train']:<10.2%} {row['test']:<10.2%} "
        f"{row['gap']:<+10.2%} {row['best_test']:<12.2%} {row['seconds']:<10.1f}"
    )

print("=" * 92)

best = max(scoreboard, key=lambda r: r['test'])
smallest_gap = min(scoreboard, key=lambda r: r['gap'])
print(f"Best final test accuracy: {best['name']} ({best['test']:.2%})")
print(f"Smallest gap:             {smallest_gap['name']} ({smallest_gap['gap']:+.2%})")

print("\nDelta vs MLP baseline (final test):")
mlp_test = scoreboard[0]['test']
for row in scoreboard[1:]:
    print(f"  {row['name']:<16} {row['test'] - mlp_test:+.2%}")


In [ ]:
names = [r['name'] for r in scoreboard]
tests = [r['test'] for r in scoreboard]
gaps = [r['gap'] for r in scoreboard]
colors = [r['color'] for r in scoreboard]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

bars = axes[0].bar(names, tests, color=colors, edgecolor='black', linewidth=0.5)
axes[0].set_ylim(0.90, 1.0)
axes[0].set_title('Final Test Accuracy')
axes[0].set_ylabel('Accuracy')
axes[0].tick_params(axis='x', rotation=15)
for bar, acc in zip(bars, tests):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
                 f"{acc:.2%}", ha='center', fontsize=10)

bars = axes[1].bar(names, gaps, color=colors, edgecolor='black', linewidth=0.5)
axes[1].set_title('Final Generalization Gap')
axes[1].set_ylabel('Train - Test')
axes[1].tick_params(axis='x', rotation=15)
axes[1].axhline(0, color='gray', linestyle=':')
for bar, gap in zip(bars, gaps):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.0005,
                 f"{gap:.2%}", ha='center', fontsize=10)

plt.tight_layout()
plt.show()


## 8. Subset vs Full Data: Context

Reminder of previous results on the 10k subset, to make the scale-up explicit.


In [ ]:
print("Reference (previous notebooks, 10k subset):")
print("  NumPy MLP:           ~86% test")
print("  PyTorch MLP:         ~96% test")
print("  CNN:                 ~98.1% / 98.25% test")
print("  CNN + L2:            ~98.1% test")
print("  CNN + Dropout:       ~98.0% test")
print()
print("This notebook (full MNIST, 60k/10k, 15 epochs):")
for row in scoreboard:
    print(f"  {row['name']:<16} {row['test']:.2%} test | gap {row['gap']:+.2%}")


## 9. Synthesis

### What scales well

1. **More data helps every model** — moving from 8k training images to 60k usually lifts test accuracy and reduces the relative value of aggressive regularization.
2. **Architecture still matters** — the CNN keeps an edge over the MLP because it respects image structure.
3. **Regularization is contextual** — L2 and Dropout were not clearly useful on the clean 10k subset. On full MNIST they may help more, help less, or mainly stabilize the gap. Judge them by test accuracy and gap together.

### Practical ranking rule

For this project, prefer:

1. highest final test accuracy
2. then smallest train/test gap
3. then training time / complexity

### Limits of this synthesis

- 15 epochs is a ranking budget, not a fully optimized training schedule
- NumPy full-batch training on 60k images is intentionally omitted (too slow for this notebook)
- No data augmentation, BatchNorm or deeper CNN variants yet


## 10. Summary

| Step in the journey | Key lesson |
|---------------------|------------|
| NumPy MLP | Understand backprop from scratch |
| PyTorch MLP | Same model, less code, mini-batches |
| CNN | Spatial inductive bias beats flat pixels |
| Regularization | L2 / Dropout are tools, not automatic wins |
| **Full MNIST** | **Scale confirms which choices still hold** |

### What's Next

- Add data augmentation
- Try BatchNorm / deeper CNNs
- Transfer learning on harder image datasets


In [ ]:
print("=" * 70)
print("SYNTHESIS SUMMARY")
print("=" * 70)
print(f"Dataset:     Full MNIST (60,000 train / 10,000 test)")
print(f"Epochs / LR: {EPOCHS} / {LR}")
print(f"Batch size:  {BATCH_SIZE}")
print("-" * 70)
for row in scoreboard:
    print(
        f"{row['name']:<16} test={row['test']:.2%} | "
        f"gap={row['gap']:+.2%} | time={row['seconds']:.1f}s"
    )
print("-" * 70)
print(f"Winner (test): {best['name']} ({best['test']:.2%})")
print(f"Best gap:      {smallest_gap['name']} ({smallest_gap['gap']:+.2%})")
print("=" * 70)
